# BUỔI 3 — Thực hành 1: CNN hai block trên FashionMNIST
Train = 6.000, validation = 1.200 (lấy từ train chính thức, seed = 42). **Không dùng tập test.**

Nội dung: in shape → huấn luyện MLP và CNN với cùng điều kiện → bảng so sánh → learning curve → confusion matrix (trên validation) → nhận xét.

In [1]:
import random, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED, EPOCHS, BATCH_SIZE = 42, 5, 64
LR, TRAIN_SIZE, VAL_SIZE = 1e-3, 6000, 1200
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

In [ ]:
def make_data():
    all_train = datasets.FashionMNIST('./data', train=True, download=True,
                                      transform=transforms.ToTensor())
    ids = torch.randperm(len(all_train), generator=torch.Generator().manual_seed(SEED)).tolist()
    return Subset(all_train, ids[:TRAIN_SIZE]), Subset(all_train, ids[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE])

def make_loaders(train_ds, val_ds):
    tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED), num_workers=0)
    va = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)
    return tr, va

class CNN2Blocks(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        return self.fc2(self.relu(self.fc1(x)))

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(784, 256),
                                 nn.ReLU(), nn.Linear(256, 10))
    def forward(self, x): return self.net(x)

## TODO 1 — In shape qua từng tầng

In [ ]:
@torch.no_grad()
def print_shapes(model):
    model.eval()
    x = torch.zeros(1, 1, 28, 28)                      # [batch, channel, H, W]
    print('input        :', tuple(x.shape))
    x = model.conv1(x);               print('sau conv1    :', tuple(x.shape))
    x = model.pool(model.relu(x));    print('sau pool1    :', tuple(x.shape))
    x = model.conv2(x);               print('sau conv2    :', tuple(x.shape))
    x = model.pool(model.relu(x));    print('sau pool2    :', tuple(x.shape))
    x = model.flatten(x);             print('trước Linear :', tuple(x.shape))

print_shapes(CNN2Blocks())

**Nhận xét:**
- Kích thước không gian giảm 28 → 14 → 7 nhờ hai lần pooling; số kênh tăng 1 → 16 → 32.
- Conv dùng `padding=1` nên chỉ pooling mới làm nhỏ ảnh. Trước Linear có 32×7×7 = 1.568 giá trị, khớp `nn.Linear(32*7*7, 128)`.

## TODO 2, 3 — Huấn luyện một epoch và validation

In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion):
    model.train()
    total_loss, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)   # nhân số mẫu để trung bình theo mẫu
        n += xb.size(0)
    return total_loss / n

@torch.inference_mode()
def validate(model, val_loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for xb, yb in val_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        out = model(xb)
        total_loss += criterion(out, yb).item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        n += xb.size(0)
    return total_loss / n, correct / n

@torch.inference_mode()
def predict_all(model, loader):
    model.eval()
    ys, ps = [], []
    for xb, yb in loader:
        ps.append(model(xb.to(DEVICE)).argmax(1).cpu()); ys.append(yb)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

## Chạy MLP và CNN với cùng seed, data split, optimizer, lr, batch, epochs

In [ ]:
set_seed()
train_ds, val_ds = make_data()
print('Device:', DEVICE, '| train:', len(train_ds), '| val:', len(val_ds))

results, histories, models = {}, {}, {}
val_loader_ref = None
for name, cls in [('MLP', MLP), ('CNN', CNN2Blocks)]:
    set_seed()
    model = cls().to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    train_loader, val_loader = make_loaders(train_ds, val_ds)
    val_loader_ref = val_loader
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'time': []}
    print(f'\n{name} | trainable_params={params}')
    for epoch in range(1, EPOCHS + 1):
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        start = time.perf_counter()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, accuracy = validate(model, val_loader, criterion)
        if DEVICE.type == 'cuda': torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        hist['train_loss'].append(train_loss); hist['val_loss'].append(val_loss)
        hist['val_acc'].append(accuracy); hist['time'].append(elapsed)
        print(f'Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, '
              f'val_acc={accuracy:.2%}, time={elapsed:.2f}s')
    histories[name], models[name] = hist, model
    results[name] = {'params': params, 'val_acc': hist['val_acc'][-1],
                     'sec_per_epoch': float(np.mean(hist['time']))}

**Nhận xét:**
- Cả hai mô hình đều giảm loss đều qua 5 epoch. MLP học nhanh hơn lúc đầu (epoch 1: 80,08% so với 76,92%), CNN đuổi kịp từ epoch 3 và vượt MLP ở epoch 5 (85,75% so với 83,08%).
- Val loss của MLP dao động (tăng nhẹ ở epoch 4), còn CNN vẫn giảm đến epoch 5, nên CNN chưa hội tụ và có thể cải thiện thêm nếu train lâu hơn.
- Thời gian CNN lúc 4s, lúc 7s do chạy trên CPU không ổn định, nên đọc số này như giá trị tham khảo.

## TODO 4 — Bảng so sánh MLP và CNN

In [ ]:
print(f"{'Mô hình':<8}{'Số tham số':>14}{'Final val acc':>16}{'Giây/epoch (TB)':>18}")
print('-' * 56)
for name, r in results.items():
    print(f"{name:<8}{r['params']:>14,}{r['val_acc']:>16.2%}{r['sec_per_epoch']:>18.2f}")

**Nhận xét:**
- Số tham số của hai mô hình gần bằng nhau (~204 nghìn so với ~207 nghìn), nên CNN tốt hơn nhờ cấu trúc (nhìn cục bộ, dùng chung kernel) chứ không phải nhờ có nhiều tham số hơn.
- CNN hơn 2,67 điểm % accuracy nhưng chậm hơn khoảng 4,6 lần mỗi epoch (5,49s so với 1,19s), vì phép tích chập tốn nhiều tính toán hơn Linear.

## Learning curve

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, EPOCHS + 1)
for name, c in [('MLP', 'tab:orange'), ('CNN', 'tab:blue')]:
    h = histories[name]
    ax[0].plot(ep, h['train_loss'], '-o', color=c, label=f'{name} train')
    ax[0].plot(ep, h['val_loss'], '--s', color=c, label=f'{name} val')
    ax[1].plot(ep, [a * 100 for a in h['val_acc']], '-o', color=c, label=name)
ax[0].set(title='Loss theo epoch', xlabel='Epoch', ylabel='Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set(title='Validation accuracy theo epoch', xlabel='Epoch', ylabel='Accuracy (%)'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[0].set_xticks(list(ep)); ax[1].set_xticks(list(ep))
plt.tight_layout(); plt.savefig('learning_curve.png', dpi=150); plt.show()

**Nhận xét:**
- Train loss và val loss của cả hai đều giảm, chưa có dấu hiệu overfit rõ rệt sau 5 epoch.
- Đường CNN bắt đầu cao hơn MLP nhưng dốc hơn và kết thúc thấp hơn ở val loss (0,4301 so với 0,4776). Train loss của MLP dao động nhẹ, đường của CNN mượt hơn.

## Ma trận nhầm lẫn (trên tập validation; hàng = nhãn thật, cột = nhãn dự đoán)

In [ ]:
def confusion(y_true, y_pred, k=10):
    cm = np.zeros((k, k), dtype=int)
    for t, p in zip(y_true, y_pred): cm[t, p] += 1
    return cm

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for a, name in zip(axes, ['MLP', 'CNN']):
    y, p = predict_all(models[name], val_loader_ref)
    cm = confusion(y, p)
    a.imshow(cm, cmap='Blues')
    a.set(title=f'Confusion matrix — {name} (val acc {(y == p).mean():.2%})',
          xlabel='Nhãn dự đoán', ylabel='Nhãn thật',
          xticks=range(10), yticks=range(10))
    a.set_xticklabels(CLASS_NAMES, rotation=45, ha='right'); a.set_yticklabels(CLASS_NAMES)
    for i in range(10):
        for j in range(10):
            a.text(j, i, cm[i, j], ha='center', va='center',
                   color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

# Các cặp lớp bị nhầm nhiều nhất của CNN
y, p = predict_all(models['CNN'], val_loader_ref)
cm = confusion(y, p); np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(-cm.ravel())[:3], cm.shape))[0]
for i, j in top: print(f'CNN nhầm {CLASS_NAMES[i]} -> {CLASS_NAMES[j]}: {cm[i, j]} mẫu')

**Nhận xét:**
- Cả hai mô hình đều nhầm nhiều ở nhóm áo: **Shirt, T-shirt, Pullover, Coat**, vì hình dạng và texture của các lớp này rất giống nhau. Giày dép, quần, túi, đầm được nhận diện tốt.
- CNN cải thiện rõ ở **Pullover** (72 mẫu đúng so với 57 của MLP, bớt nhầm sang Coat) và các lớp giày dép. Lớp **Shirt** vẫn yếu ở cả hai (53 mẫu đúng ở CNN, 50 ở MLP), hay bị nhầm thành T-shirt (34 mẫu).

## TODO 5 — Số liệu để viết nhận xét

In [ ]:
m, c = results['MLP'], results['CNN']
print(f"- Tham số: MLP {m['params']:,} vs CNN {c['params']:,} (CNN gấp {c['params']/m['params']:.2f} lần MLP).")
print(f"- Val acc cuối: MLP {m['val_acc']:.2%} vs CNN {c['val_acc']:.2%} (chênh {100*(c['val_acc']-m['val_acc']):+.2f} điểm %).")
print(f"- Thời gian/epoch: MLP {m['sec_per_epoch']:.2f}s vs CNN {c['sec_per_epoch']:.2f}s.")
print(f"- Train loss cuối: MLP {histories['MLP']['train_loss'][-1]:.4f}, CNN {histories['CNN']['train_loss'][-1]:.4f}.")

### Nhận xét tổng hợp
CNN đạt val accuracy 85,75%, cao hơn MLP (83,08%) khoảng 2,67 điểm %, trong khi số tham số gần như bằng nhau (206.922 so với 203.530), tức là lợi thế đến từ việc khai thác quan hệ không gian giữa các pixel (nhìn cục bộ và dùng chung kernel). Đổi lại, CNN chậm hơn khoảng 4,6 lần mỗi epoch (5,49s so với 1,19s trên CPU). Tensor đi qua mạng theo đúng thiết kế: 1×28×28 → 16×28×28 → 16×14×14 → 32×14×14 → 32×7×7 → 1.568 → 10, khớp với đầu vào của Linear. Cả hai mô hình đều nhầm nhiều giữa các lớp áo giống nhau (Shirt, T-shirt, Pullover, Coat). Lưu ý kết quả chỉ từ một seed, 5 epoch và 1.200 mẫu validation nên mức chênh 2,67 điểm % là gợi ý chứ chưa phải kết luận chắc chắn.